# 08 - M8: training loop (real CLIP, real T4)
**Runtime > T4 GPU.** Gate (contract): (1) overfits 4 images, (2) resumes from a checkpoint, (3) fp16 stable.
Real pipeline throughout (CLIP ViT-L/14@336 + LoRA, 2 steered Mamba, concept readout), `src/train.py` Trainer, `controlled` protocol, fold Midjourney. Nothing here touches the held-out test split.
Budget: peak < 10 GiB at batch 8.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import json, time, os, gc, shutil, numpy as np, torch
from datasets import load_dataset
from transformers import CLIPVisionModel, CLIPTextModelWithProjection, CLIPTokenizer
from src.data import load_manifest, ManifestDataset
from src import concepts as K
from src.train import TrainConfig, Trainer, EpochSeeded, make_model
dev = 'cuda'
res = {'milestone': 'M8'}
ASSET = 'assets/concepts_text_768.npy'
if os.path.exists(ASSET):
    T = K.load_or_encode(ASSET)
else:
    tm = CLIPTextModelWithProjection.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32)
    T = K.load_or_encode(ASSET, tm, CLIPTokenizer.from_pretrained(K.TEXT_CHECKPOINT), device='cuda'); del tm
mf = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=mf['revision'])
tr = ManifestDataset(mf, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)  # no flip for the gates
labels = np.asarray(ds['train']['label'])[tr.indices]                                                    # column read, no image decode
pos = [int(i) for i in np.where(labels == 0)[0][:2]] + [int(i) for i in np.where(labels == 1)[0][:2]]
print('4 gate images (2 real, 2 fake), labels:', labels[pos].tolist())
RUNS = f'{DRIVE_ROOT}/runs/m8'; shutil.rmtree(RUNS, ignore_errors=True); os.makedirs(RUNS)

def new_trainer(cfg, name, train_ds, val_ds=None):
    clip = CLIPVisionModel.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32).eval()
    m = make_model(clip, T, cfg)
    print(name, m.param_report())
    return Trainer(m, train_ds, val_ds, cfg, f'{RUNS}/{name}', device=dev)

def release(t):
    del t.model, t; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# GATE 1 - overfit 4 images. fp16 autocast on, steered arm, constant LR, no flip, 1 batch of 4 per epoch.
c1 = TrainConfig(arm='steered', protocol='controlled', epochs=60, batch_size=4, lr_lora=3e-4, lr_new=1e-3,
                 schedule='const', warmup_frac=0.05, num_workers=0, ckpt_every=1000, log_every=10, diag_every=20)
t = new_trainer(c1, 'overfit4', EpochSeeded(tr, 0, positions=pos))
torch.cuda.reset_peak_memory_stats()
t.fit()
L = t.step_losses
print('loss first/last5:', np.mean(L[:5]), np.mean(L[-5:]), '| skipped', t.state['skipped'], '| nonfinite', t.state['nonfinite'])
xb = torch.stack([tr[i][0] for i in pos]).to(dev); yb = torch.tensor([float(tr[i][1]) for i in pos])
t.model.eval()
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    lg = t.model(xb)['logit'].float().cpu()
print('logits', lg.tolist(), 'labels', yb.tolist())
g1 = {'loss_first5': float(np.mean(L[:5])), 'loss_last5': float(np.mean(L[-5:])), 'logits': lg.tolist(), 'labels': yb.tolist(),
      'peak_gib': torch.cuda.max_memory_allocated() / 2**30}
assert np.mean(L[-5:]) < 0.05, 'did not overfit'
assert ((lg > 0).float() == yb).all() and t.state['nonfinite'] == 0
res['gate_overfit4'] = g1; print('GATE 1 OK'); release(t)

In [ ]:
# GATE 2 - resume, mid-epoch, with lost steps. 8 images (4 real, 4 fake), batch 2 -> 4 batches/epoch, 3 epochs = 12 steps,
# checkpoint every 5 steps. Run A: 12 steps straight. Run B: killed at step 7 (last checkpoint = step 5, mid-epoch 2, so steps 6-7
# are lost), then a FRESH model + Trainer resumes from last.pt.
# Pass criteria: (i) the restored state is EXACT: weights bitwise equal the checkpoint, counters (step 5, epoch 1, batch 1);
# (ii) the resumed trajectory (steps 6-12) matches A's within the run-to-run noise (GPU fp16 kernels are not bitwise
# deterministic: A and B differ before the kill too, so that prefix difference is the noise floor we compare against).
pos8 = [int(i) for i in np.where(labels == 0)[0][:4]] + [int(i) for i in np.where(labels == 1)[0][:4]]
c2 = TrainConfig(arm='steered', protocol='controlled', epochs=3, batch_size=2, lr_lora=3e-4, lr_new=1e-3,
                 schedule='cosine', warmup_frac=0.1, num_workers=0, ckpt_every=5, log_every=4, diag_every=1000)
mk = lambda name: new_trainer(c2, name, EpochSeeded(tr, 0, positions=pos8))
A = mk('resume_A'); A.fit(); LA = list(A.step_losses); SA = A.model.trainable_state_dict(); assert len(LA) == 12; release(A)
B = mk('resume_B'); B.fit(stop_after_steps=7); assert not B.state['done']; LB1 = list(B.step_losses); release(B)   # 'disconnect'
ck = torch.load(f'{RUNS}/resume_B/last.pt', map_location='cpu', weights_only=True)
st = ck['state']; print('checkpoint on disk:', {k: st[k] for k in ('step', 'epoch', 'batch')})
assert (st['step'], st['epoch'], st['batch']) == (5, 1, 1)
B2 = mk('resume_B'); B2.load()
exact = all(torch.equal(v.cpu(), ck['model'][k]) for k, v in B2.model.trainable_state_dict().items())
assert exact and B2.state['step'] == 5, 'restored state is not identical to the checkpoint'
B2.fit(); assert B2.state['done'] and B2.state['step'] == 12
LB2 = list(B2.step_losses); SB = B2.model.trainable_state_dict(); assert len(LB2) == 7     # steps 6..12
noise = max(abs(a - b) for a, b in zip(LA[:7], LB1))        # same code, same data, two uninterrupted processes
dev_ = max(abs(a - b) for a, b in zip(LA[5:], LB2))
pdiff = max((SA[k].float() - SB[k].float()).abs().max().item() for k in SA)
print(f'restored state bitwise exact: {exact} | loss diff, uninterrupted prefix (noise floor): {noise:.2e} | resumed segment: {dev_:.2e} | max param diff {pdiff:.2e}')
assert dev_ < max(0.02, 5 * noise) and pdiff < 0.01
res['gate_resume'] = {'restored_state_bitwise_exact': exact, 'noise_floor_loss_diff': noise, 'resumed_loss_diff': dev_, 'max_param_diff': pdiff, 'resumed_steps': len(LB2)}
print('GATE 2 OK'); release(B2)

In [ ]:
# GATE 3 - fp16 stability on real shuffled training data: batch 8, fp16 autocast + GradScaler, 40 steps, workers 2.
c3 = TrainConfig(arm='steered', protocol='controlled', epochs=1, batch_size=8, num_workers=2, ckpt_every=20, log_every=5, diag_every=20)
tr_flip = ManifestDataset(mf, 'train', ds['train'], ds['validation'], protocol='controlled', train=True)
t = new_trainer(c3, 'fp16', EpochSeeded(tr_flip, 0, positions=np.arange(320)))
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
t.fit(); sec = time.time() - t0
L = t.step_losses; peak = torch.cuda.max_memory_allocated() / 2**30
g3 = {'steps': len(L), 'loss_first10': float(np.mean(L[:10])), 'loss_last10': float(np.mean(L[-10:])), 'skipped_steps': t.state['skipped'],
      'nonfinite_steps': t.state['nonfinite'], 'peak_gib': peak, 'img_per_s': 320 / sec,
      'final_scale': t.scaler.get_scale(), 'all_params_finite': all(torch.isfinite(v).all().item() for v in t.model.trainable_state_dict().values())}
print(json.dumps(g3, indent=1))
assert g3['nonfinite_steps'] == 0 and g3['skipped_steps'] <= 5 and g3['all_params_finite'] and np.isfinite(L).all() and peak < 10
res['gate_fp16'] = g3
diag = [json.loads(l) for l in open(t.log_path) if '"diag"' in l][-1]['diag']
print('last diag: alpha', diag['alpha'], '\nmix', diag['mix_weights'], 'scale', diag['affinity_scale'], '\ngates', diag['gates'])
print('GATE 3 OK'); release(t)

In [ ]:
res['gate'] = 'PASS'
for p in (f'{DRIVE_ROOT}/runs/m8_gate.json', '/content/csm-clip/runs/m8_gate.json'):
    json.dump(res, open(p, 'w'), indent=1)
print(json.dumps(res, indent=1))
# Then: commit runs/m8_gate.json, set M8 to Done in docs/contract.md, tag m8-done.